# Double Fault Dissimilarity entre Classificadores\n\nEm vez de t-SNE amostral, cada **classificador** é colocado como um ponto (x, y) calculado via **MDS** sobre a matriz de dissimilaridade **Double Fault**.\n\n### Double Fault Measure\n\nDado dois classificadores *i* e *j* e *N* amostras de validação:\n\n| | j acerta | j erra |\n|---|---|---|\n| **i acerta** | N¹¹ | N¹⁰ |\n| **i erra** | N⁰¹ | **N⁰⁰** |\n\n**DF(i, j) = N⁰⁰ / N** — fração de amostras em que **ambos falham simultaneamente**.\n\n- DF alto → falhas compartilhadas → baixa diversidade → pontos distantes no MDS\n- DF baixo → falhas complementares → alta diversidade → pontos próximos no MDS\n\nA matriz DF é usada diretamente como dissimilaridade no MDS 2D."
  

## 1. Imports & Setup

In [ ]:
import os, sys, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import DataLoader, SequentialSampler
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    efficientnet_v2_s, EfficientNet_V2_S_Weights,
    efficientnet_b3, EfficientNet_B3_Weights,
    swin_t, Swin_T_Weights,
)
from sklearn.manifold import MDS
from sklearn.metrics import cohen_kappa_score

sys.path.append('../..')
from utils.dataset import PandasDataset
from utils.mil import PandasWithMilDataset, EfficientNetMIL, SwinMIL
from utils.models import EfficientNetApi

print('Imports OK')

## 2. Configuração

In [2]:
SEED           = 42
NUM_WORKERS    = 4
OUTPUT_CLASSES = 5
BATCH_PATCH    = 4
BATCH_MIL      = 8
MAX_PATCHES    = 36
PCA_COMPONENTS = 50
TSNE_PERPLEXITY = 40
TSNE_ITER       = 1000

DROPOUT_B0    = 0.6
DROPOUT_V2S   = 0.4422
DROPOUT_B3    = 0.6
DROPOUT_SWA   = 0.6
DROPOUT_MIL   = 0.4
DROPOUT_SWMIL = 0.4
UNFREEZE_V2S  = 3

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

ROOT_DIR    = '../..'
DATA_DIR    = '../../..'
IMAGES_DIR  = os.path.join(DATA_DIR, 'tiles')
PATCHES_DIR = '/home/woshington/Projects/Doutorado/bag_of_patches'

CKPT_B0    = '../tests/baseline/models/b0-entropy-ordinal-3-focal.pth'
CKPT_V2S   = '../tests/baseline/models/v2-optuna-ordinal-focal.pth'
CKPT_B3    = '../tests/baseline/models/b3-entropy-ordinal-focal.pth'
CKPT_SWA   = '../tests/baseline/models/b0-entropy-ordinal-swa.pth'
CKPT_MIL   = '../tests/transformers/models/b0-mil-focal.pth'
CKPT_SWMIL = '../tests/transformers/models/swin-t-mil-optuna.pth'

os.makedirs('logs', exist_ok=True)
print(f'Device: {DEVICE}')

Device: cuda


## 3. EfficientNetV2Api

In [3]:
class EfficientNetV2Api(nn.Module):
    def __init__(self, model, output_dimensions, dropout_rate=0.4, unfreeze_blocks=2):
        super().__init__()
        self.model = model
        for param in self.model.parameters():
            param.requires_grad = False
        if hasattr(self.model, 'features') and unfreeze_blocks > 0:
            for block in self.model.features[-unfreeze_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True
        if isinstance(self.model.classifier, nn.Sequential):
            in_features = self.model.classifier[-1].in_features
        else:
            in_features = self.model.classifier.in_features
        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, output_dimensions),
        )

    def extract(self, x):
        x = self.model(x)
        if x.ndim == 4:
            x = x.mean(dim=[2, 3])
        return x

    def forward(self, x):
        return self.head(self.extract(x))

print('EfficientNetV2Api OK')

EfficientNetV2Api OK


## 4. Carregamento dos Modelos

In [4]:
def _load(label, ctor, ckpt):
    m = ctor()
    m.load_state_dict(torch.load(ckpt, weights_only=True))
    return m.to(DEVICE).eval()

model_b0 = _load('B0', lambda: EfficientNetApi(
    efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT),
    OUTPUT_CLASSES, DROPOUT_B0), CKPT_B0)
print('  B0 OK')

model_v2s = _load('V2S', lambda: EfficientNetV2Api(
    efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT),
    OUTPUT_CLASSES, DROPOUT_V2S, UNFREEZE_V2S), CKPT_V2S)
print('  V2S OK')

model_b3 = _load('B3', lambda: EfficientNetApi(
    efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT),
    OUTPUT_CLASSES, DROPOUT_B3), CKPT_B3)
print('  B3 OK')

model_swa = _load('SWA', lambda: EfficientNetApi(
    efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT),
    OUTPUT_CLASSES, DROPOUT_SWA), CKPT_SWA)
print('  SWA OK')

model_mil = _load('MIL-B0', lambda: EfficientNetMIL(
    efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT),
    output_classes=OUTPUT_CLASSES, dropout_rate=DROPOUT_MIL,
    hidden_dim=512, gated=True, pool='att'), CKPT_MIL)
print('  MIL-B0 OK')

model_swmil = _load('SwinMIL', lambda: SwinMIL(
    swin_t(weights=Swin_T_Weights.DEFAULT),
    output_classes=OUTPUT_CLASSES, unfreeze_last_blocks=2,
    dropout_rate=DROPOUT_SWMIL, hidden_dim=512, gated=True, pool='att'), CKPT_SWMIL)
print('  SwinMIL OK')

  B0 OK
  V2S OK
  B3 OK
  SWA OK
  MIL-B0 OK
  SwinMIL OK


## 5. Dados de Validação

In [5]:
df_all     = pd.read_csv(f'{ROOT_DIR}/data/train_5fold.csv')
df_entropy = pd.read_csv(f'{ROOT_DIR}/data/entropy.csv')
df_all.columns     = df_all.columns.str.strip()
df_entropy.columns = df_entropy.columns.str.strip()

hard_ids = set(
    df_entropy.sort_values('difficulty_score', ascending=False)
              .head(int(len(df_entropy) * 0.2))['image_id']
)
df_all = df_all[~df_all['image_id'].isin(hard_ids)].reset_index(drop=True)
df_val = df_all[df_all['fold'] == 3].reset_index(drop=True)

def filter_patch(df):
    m = df['image_id'].apply(lambda x: os.path.isfile(os.path.join(IMAGES_DIR, f'{x}.png')))
    return df[m].reset_index(drop=True)

def filter_bags(df):
    m = df['image_id'].apply(lambda x: os.path.isdir(os.path.join(PATCHES_DIR, str(x))))
    return df[m].reset_index(drop=True)

df_val_patch = filter_patch(df_val)
df_val_mil_  = filter_bags(df_val)
common = set(df_val_patch['image_id']) & set(df_val_mil_['image_id'])
df_val_c = df_val_patch[df_val_patch['image_id'].isin(common)].reset_index(drop=True)
print(f'Validacao (patch+MIL): {len(df_val_c)} amostras')
print(df_val_c['isup_grade'].value_counts().sort_index().to_string())

Validacao (patch+MIL): 1767 amostras
isup_grade
0    485
1    449
2    228
3    204
4    206
5    195


## 6. Inferência: Probabilidades e Features

In [ ]:
def get_patch_probs(model, df, images_dir, batch_size, device, desc=''):
    """Retorna probs (N,5), targets (N,), img_ids."""
    ds = PandasDataset(images_dir, df, transforms=None, format='png')
    dl = DataLoader(ds, batch_size=batch_size, num_workers=NUM_WORKERS,
                    sampler=SequentialSampler(ds), pin_memory=True)
    probs_l, tgt_l, id_l = [], [], []
    model.eval()
    with torch.no_grad():
        for bx, by, bids in tqdm(dl, desc=desc, leave=False):
            probs_l.append(torch.sigmoid(model(bx.to(device))).cpu().numpy())
            tgt_l.extend(by.sum(1).long().tolist())
            id_l.extend([str(i) for i in bids])
    return np.vstack(probs_l), np.array(tgt_l), id_l


def get_mil_probs(model, df, patches_dir, batch_size, device, desc=''):
    """Retorna probs (N,5), targets (N,), img_ids."""
    ds = PandasWithMilDataset(patches_dir, df, transforms=None, max_patches=MAX_PATCHES)
    dl = DataLoader(ds, batch_size=batch_size, num_workers=NUM_WORKERS,
                    sampler=SequentialSampler(ds), pin_memory=True)
    probs_l, tgt_l, id_l = [], [], []
    model.eval()
    with torch.no_grad():
        for bag, mask, tgts, bids in tqdm(dl, desc=desc, leave=False):
            out = model(bag.to(device), mask.to(device))
            probs_l.append(torch.sigmoid(out['logits']).cpu().numpy())
            tgt_l.append(tgts.sum(1).long().numpy())
            id_l.extend([str(i) for i in bids])
    return np.vstack(probs_l), np.concatenate(tgt_l), id_l


def align_probs(data_dict):
    """Alinha por image_id. Returns: ids, targets, {name: probs}."""
    names = list(data_dict.keys())
    frames = {}
    for name, (probs, tgts, ids) in data_dict.items():
        df_m = pd.DataFrame({'target': tgts, 'row_idx': np.arange(len(tgts))}, index=ids)
        df_m = df_m[~df_m.index.duplicated(keep='first')]
        df_m['probs'] = [probs[i] for i in df_m['row_idx']]
        frames[name] = df_m
    common = frames[names[0]].index
    for n in names[1:]:
        common = common.intersection(frames[n].index)
    ref_tgts, aligned = None, {}
    for n in names:
        sub = frames[n].loc[common]
        aligned[n] = np.vstack(sub['probs'].tolist())
        if ref_tgts is None:
            ref_tgts = sub['target'].values.astype(int)
    print(f'Amostras comuns: {len(common)}')
    return list(common), ref_tgts, aligned

print('Funcoes OK')

In [ ]:
print('Inferencia na validacao...')
raw = {
    'B0':      get_patch_probs(model_b0,   df_val_c, IMAGES_DIR, BATCH_PATCH, DEVICE, 'B0'),
    'V2S':     get_patch_probs(model_v2s,  df_val_c, IMAGES_DIR, BATCH_PATCH, DEVICE, 'V2S'),
    'B3':      get_patch_probs(model_b3,   df_val_c, IMAGES_DIR, BATCH_PATCH, DEVICE, 'B3'),
    'SWA':     get_patch_probs(model_swa,  df_val_c, IMAGES_DIR, BATCH_PATCH, DEVICE, 'SWA'),
    'MIL-B0':  get_mil_probs  (model_mil,  df_val_c, PATCHES_DIR, BATCH_MIL,  DEVICE, 'MIL-B0'),
    'SwinMIL': get_mil_probs  (model_swmil,df_val_c, PATCHES_DIR, BATCH_MIL,  DEVICE, 'SwinMIL'),
}

val_ids, val_targets, all_probs = align_probs(raw)
MODEL_NAMES = list(all_probs.keys())
M           = len(MODEL_NAMES)

# Hard predictions (ordinal decoding)
all_preds = {n: (all_probs[n] > 0.5).sum(axis=1) for n in MODEL_NAMES}

# Error mask per model: True where classifier is wrong
all_errors = {n: all_preds[n] != val_targets for n in MODEL_NAMES}

N = len(val_targets)
print(f'\nN={N} amostras | {M} classificadores')
for n in MODEL_NAMES:
    acc = (~all_errors[n]).mean()
    err = all_errors[n].mean()
    print(f'  {n:<10}  acc={acc*100:.2f}%  err={err*100:.2f}%')

## 7. Matriz Double Fault\n\n`DF(i, j) = N⁰⁰ / N` — fração de amostras em que os classificadores *i* **e** *j* ambos erram.

In [ ]:
def double_fault(preds_i, preds_j, targets):
    """DF(i,j) = fraction of samples both classifiers misclassify."""
    return ((preds_i != targets) & (preds_j != targets)).sum() / len(targets)


# ── Build full DF matrix (M x M) ──────────────────────────────────────
df_mat = np.zeros((M, M))
for i, ni in enumerate(MODEL_NAMES):
    for j, nj in enumerate(MODEL_NAMES):
        df_mat[i, j] = double_fault(all_preds[ni], all_preds[nj], val_targets)
# Diagonal = individual error rate (both = same classifier)
# Set to 0 for MDS (self-distance = 0)
df_dissim = df_mat.copy()
np.fill_diagonal(df_dissim, 0.0)

# ── Print table ───────────────────────────────────────────────────────
print('Double Fault Matrix (DF values)')
print(f'{"":>10}' + ''.join(f'{n:>10}' for n in MODEL_NAMES))
for i, ni in enumerate(MODEL_NAMES):
    row = ''.join(f'{df_mat[i,j]:>10.4f}' for j in range(M))
    print(f'{ni:>10}{row}')

# ── Heatmap ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: raw DF (including diagonal = individual error rate)
sns.heatmap(df_mat, annot=True, fmt='.4f', cmap='YlOrRd',
            xticklabels=MODEL_NAMES, yticklabels=MODEL_NAMES,
            ax=axes[0], linewidths=0.5, vmin=0)
axes[0].set_title('DF(i,j) — Double Fault Matrix\n(diagonal = taxa de erro individual)', fontsize=11)
axes[0].tick_params(axis='x', rotation=30)

# Right: off-diagonal only (dissimilarity for MDS)
mask_diag = np.eye(M, dtype=bool)
sns.heatmap(df_dissim, annot=True, fmt='.4f', cmap='YlOrRd',
            xticklabels=MODEL_NAMES, yticklabels=MODEL_NAMES,
            ax=axes[1], mask=mask_diag, linewidths=0.5, vmin=0)
axes[1].set_title('Dissimilaridade DF (diagonal=0)\nusada como entrada do MDS', fontsize=11)
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Double Fault Dissimilarity entre Classificadores', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('logs/df-matrix.png', dpi=200, bbox_inches='tight')
plt.show()

## 8. MDS: Coordenadas (x, y) por Classificador\n\nO MDS (Multidimensional Scaling) projeta a matriz 6×6 de dissimilaridade DF em 2D, preservando as distâncias relativas entre classificadores.

In [ ]:
mds = MDS(n_components=2, dissimilarity='precomputed',
          normalized_stress='auto', random_state=SEED, n_init=10)
coords = mds.fit_transform(df_dissim)   # (M, 2)

print('Coordenadas MDS dos classificadores:')
print(f'  {"Modelo":<10}   x         y')
for i, name in enumerate(MODEL_NAMES):
    print(f'  {name:<10}  {coords[i,0]:+.6f}  {coords[i,1]:+.6f}')
print(f'\nStress (MDS): {mds.stress_:.6f}')

In [ ]:
CNN_NAMES = ['B0', 'V2S', 'B3', 'SWA']
MIL_NAMES = ['MIL-B0', 'SwinMIL']
COLOR_CNN = '#3498db'
COLOR_MIL = '#e74c3c'

acc_per_model = {n: (~all_errors[n]).mean() for n in MODEL_NAMES}
err_per_model = {n: all_errors[n].mean()    for n in MODEL_NAMES}

# ── Node size proportional to accuracy ────────────────────────────────
min_acc = min(acc_per_model.values())
max_acc = max(acc_per_model.values())
size_scale = lambda a: 300 + 1200 * (a - min_acc) / max(max_acc - min_acc, 1e-6)

# ── Edge colormap: dark red = high DF (many shared failures) ──────────
df_vals_offdiag = df_dissim[df_dissim > 0]
vmin_edge = df_vals_offdiag.min()
vmax_edge = df_vals_offdiag.max()
edge_cmap = cm.get_cmap('Reds')

fig, ax = plt.subplots(figsize=(9, 8))

# ── Draw edges: one line per pair, opacity/thickness proportional to DF
for i in range(M):
    for j in range(i + 1, M):
        df_val = df_dissim[i, j]
        norm_df = (df_val - vmin_edge) / max(vmax_edge - vmin_edge, 1e-6)
        lw    = 1.0 + 5.0 * norm_df
        alpha = 0.25 + 0.55 * norm_df
        color = edge_cmap(0.3 + 0.7 * norm_df)
        ax.plot([coords[i, 0], coords[j, 0]],
                [coords[i, 1], coords[j, 1]],
                '-', color=color, linewidth=lw, alpha=alpha, zorder=1)
        # DF label at midpoint
        mx = (coords[i, 0] + coords[j, 0]) / 2
        my = (coords[i, 1] + coords[j, 1]) / 2
        ax.text(mx, my, f'{df_val:.4f}',
                fontsize=7.5, ha='center', va='center',
                color='#555555', zorder=3,
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.7))

# ── Draw nodes ────────────────────────────────────────────────────────
for i, name in enumerate(MODEL_NAMES):
    color = COLOR_CNN if name in CNN_NAMES else COLOR_MIL
    acc   = acc_per_model[name]
    sz    = size_scale(acc)
    ax.scatter(coords[i, 0], coords[i, 1],
               c=color, s=sz, zorder=4,
               edgecolors='white', linewidths=2.0)
    # Label with name + accuracy
    offset_x = 0.002 * (coords[:, 0].max() - coords[:, 0].min())
    offset_y = 0.04  * (coords[:, 1].max() - coords[:, 1].min())
    ax.annotate(
        f'{name}\nacc={acc*100:.1f}%',
        xy=(coords[i, 0], coords[i, 1]),
        xytext=(coords[i, 0] + offset_x, coords[i, 1] + offset_y),
        fontsize=9, ha='center', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=color, alpha=0.85, lw=1.2),
        zorder=5,
    )

# ── Legend and colorbar ───────────────────────────────────────────────
patch_cnn = mpatches.Patch(color=COLOR_CNN, label='CNN patch-level')
patch_mil = mpatches.Patch(color=COLOR_MIL, label='MIL')
size_note  = mpatches.Patch(color='none', label='Tamanho ∝ accuracy')
ax.legend(handles=[patch_cnn, patch_mil, size_note],
          loc='lower right', fontsize=9, framealpha=0.85)

sm = cm.ScalarMappable(cmap=edge_cmap,
                       norm=plt.Normalize(vmin=vmin_edge, vmax=vmax_edge))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('DF(i,j)  —  dupla falha simultânea', fontsize=9)

ax.set_title(
    'Dispersão entre Classificadores via Double Fault (MDS 2D)\n'
    'Distância ∝ DF — classificadores que falham juntos ficam mais distantes',
    fontsize=11, pad=12,
)
ax.axis('off')
plt.tight_layout()
plt.savefig('logs/df-mds-classifiers.png', dpi=200, bbox_inches='tight')
plt.show()

## 9. Double Fault por Grau ISUP\n\nRepete o cálculo do DF para cada subconjunto de amostras de um grau ISUP — mostra em qual grau os modelos mais falham juntos.

In [ ]:
ISUP_LABELS = [f'ISUP {g}' for g in range(6)]

# ── Per-ISUP DF matrices ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for g, ax in enumerate(axes.flat):
    mask = val_targets == g
    n_g  = mask.sum()
    if n_g == 0:
        ax.axis('off')
        continue

    mat_g = np.zeros((M, M))
    for i, ni in enumerate(MODEL_NAMES):
        for j, nj in enumerate(MODEL_NAMES):
            mat_g[i, j] = double_fault(all_preds[ni][mask],
                                       all_preds[nj][mask],
                                       val_targets[mask])

    # MDS for this grade
    dissim_g = mat_g.copy()
    np.fill_diagonal(dissim_g, 0.0)

    # Heatmap
    sns.heatmap(dissim_g, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=MODEL_NAMES, yticklabels=MODEL_NAMES,
                ax=ax, mask=np.eye(M, dtype=bool),
                vmin=0, linewidths=0.4, annot_kws={'size': 8},
                cbar=False)
    ax.set_title(f'{ISUP_LABELS[g]}  (n={n_g})', fontsize=10)
    ax.tick_params(labelsize=7, rotation=30)

plt.suptitle('Double Fault por Grau ISUP (submatriz off-diagonal)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('logs/df-per-isup.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── MDS per ISUP: um grafico de dispersao para cada grau ──────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for g, ax in enumerate(axes.flat):
    mask = val_targets == g
    n_g  = mask.sum()
    if n_g < 2:
        ax.axis('off')
        continue

    mat_g = np.zeros((M, M))
    for i, ni in enumerate(MODEL_NAMES):
        for j, nj in enumerate(MODEL_NAMES):
            mat_g[i, j] = double_fault(all_preds[ni][mask],
                                       all_preds[nj][mask],
                                       val_targets[mask])
    dissim_g = mat_g.copy()
    np.fill_diagonal(dissim_g, 0.0)

    try:
        mds_g = MDS(n_components=2, dissimilarity='precomputed',
                    normalized_stress='auto', random_state=SEED, n_init=10)
        c_g = mds_g.fit_transform(dissim_g)
    except Exception:
        ax.axis('off')
        continue

    df_off = dissim_g[dissim_g > 0]
    vmin_g  = df_off.min() if len(df_off) else 0
    vmax_g  = df_off.max() if len(df_off) else 1

    for i in range(M):
        for j in range(i + 1, M):
            dv = dissim_g[i, j]
            nv = (dv - vmin_g) / max(vmax_g - vmin_g, 1e-6)
            ax.plot([c_g[i,0], c_g[j,0]], [c_g[i,1], c_g[j,1]],
                    '-', color=edge_cmap(0.3 + 0.7*nv),
                    linewidth=1.0 + 4.0*nv, alpha=0.4 + 0.4*nv, zorder=1)

    for i, name in enumerate(MODEL_NAMES):
        color  = COLOR_CNN if name in CNN_NAMES else COLOR_MIL
        err_g  = all_errors[name][mask].mean()
        ax.scatter(c_g[i, 0], c_g[i, 1], c=color, s=200, zorder=4,
                   edgecolors='white', linewidths=1.5)
        ax.annotate(f'{name}\n(err={err_g:.2f})',
                    xy=(c_g[i,0], c_g[i,1]),
                    xytext=(c_g[i,0], c_g[i,1] + 0.04*(c_g[:,1].ptp() or 1)),
                    fontsize=7, ha='center',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white',
                              ec=color, alpha=0.8, lw=0.8),
                    zorder=5)

    ax.set_title(f'{ISUP_LABELS[g]}  (n={n_g})', fontsize=10)
    ax.axis('off')

plt.suptitle('Dispersao Double Fault MDS por Grau ISUP', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('logs/df-mds-per-isup.png', dpi=200, bbox_inches='tight')
plt.show()

## 10. Ranking de Pares por Double Fault

In [ ]:
pairs = []
for i in range(M):
    for j in range(i + 1, M):
        ni, nj = MODEL_NAMES[i], MODEL_NAMES[j]
        df_val  = df_dissim[i, j]
        n00     = int(df_val * N)
        n10     = int((all_errors[ni] & ~all_errors[nj]).sum())
        n01     = int((~all_errors[ni] & all_errors[nj]).sum())
        n11     = int((~all_errors[ni] & ~all_errors[nj]).sum())
        pairs.append((ni, nj, df_val, n00, n10, n01, n11))

pairs.sort(key=lambda x: -x[2])   # descending DF

print(f'{"Par":<22} {"DF":>7} {"N00":>6} {"N10":>6} {"N01":>6} {"N11":>6}')
print('-' * 55)
for ni, nj, dv, n00, n10, n01, n11 in pairs:
    print(f'{ni+" vs "+nj:<22} {dv:>7.4f} {n00:>6} {n10:>6} {n01:>6} {n11:>6}')

# ── Bar chart ─────────────────────────────────────────────────────────
labels_pairs = [f'{a}\nvs\n{b}' for a, b, *_ in pairs]
df_vals      = [x[2] for x in pairs]
bar_colors   = ['#e74c3c' if (a in MIL_NAMES) != (b in MIL_NAMES)
                else ('#3498db' if a in CNN_NAMES else '#e67e22')
                for a, b, *_ in pairs]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(pairs)), df_vals, color=bar_colors, edgecolor='white')
ax.set_xticks(range(len(pairs)))
ax.set_xticklabels(labels_pairs, fontsize=8, va='top')
ax.set_ylabel('DF(i,j) — Double Fault', fontsize=10)
ax.set_title('Ranking de Pares por Double Fault\n'
             'Azul: CNN×CNN  |  Laranja: MIL×MIL  |  Vermelho: CNN×MIL', fontsize=11)
ax.grid(axis='y', alpha=0.35)
for bar, v in zip(bars, df_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.001,
            f'{v:.4f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('logs/df-ranking-pairs.png', dpi=200, bbox_inches='tight')
plt.show()

## 11. Sumário

In [ ]:
print('=' * 60)
print('  Double Fault Dissimilarity — Sumario (validacao)')
print('=' * 60)

print('\n[Accuracy individual]')
for n in MODEL_NAMES:
    print(f'  {n:<10}  acc={acc_per_model[n]*100:.2f}%  err={err_per_model[n]*100:.2f}%')

most_sim = pairs[-1]
most_dis = pairs[0]
print(f'\n[Par com menor DF (mais complementares / diversos)]')
print(f'  {most_sim[0]} vs {most_sim[1]}  DF={most_sim[2]:.4f}  N00={most_sim[3]}')

print(f'\n[Par com maior DF (mais falhas compartilhadas)]')
print(f'  {most_dis[0]} vs {most_dis[1]}  DF={most_dis[2]:.4f}  N00={most_dis[3]}')

print(f'\n[MDS stress: {mds.stress_:.6f}]')

print('\nArquivos salvos:')
for f in sorted(os.listdir('logs')):
    if f.startswith('df-'):
        print(f'  logs/{f}')